# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _One item/ line within an order._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
display(lines)

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

Answer: Vendor V-07 did not match the vendor list, and it has 3 units attached to it (3 Hot Dogs).

In [3]:
# TODO
vendors_df = pd.json_normalize(vendors_json)
joined = lines.merge(
    vendors_df,
    on='vendor_id',
    how='left',
    indicator=True
)
unmatched = joined[joined['_merge'] == 'left_only']
print(unmatched.groupby('vendor_id')['qty'].sum())
display(joined)

vendor_id
V-07    3
Name: qty, dtype: int64


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
joined['name'] = joined['name'].fillna('Unknown Vendor')
vendor_totals = joined.groupby('name')['qty'].sum()
display(vendor_totals)

# The unmatched vendor appears in this table as an Unknown Vendor because V-07 is missing from the vendor list, so there's no corresponding name.


,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown Vendor,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO

zone_totals = joined.groupby('zone')['qty'].sum()
display(zone_totals)
print("zone with the most items:", zone_totals.idxmax())
print("total units by zone:", zone_totals.sum())

# The total is 4 units in Zone A. The remaining 3 units belong to vendor V-07, which has no assigned zone because it was not in the vendor list. Those 3 units were excluded from the zone totals.

,qty
zone,
A,4


zone with the most items: A
total units by zone: 4


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO

for order in ragged_json:
    order.setdefault  ('lines', [])
ragged_df = pd.json_normalize(
    ragged_json, record_path='lines', meta=['order', 'vendor_id'])
display(ragged_df)

# missing qty are nan and not 0 because we don't actually know how many fries were ordered.

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a. Having one row per line item makes it easier to look at each item separately and calculate quantities. Like, we can easily find how many items each vendor sold. If we had one row per order, all the items would be grouped together, making it harder to calculate totals for each vendor.

b. A missing quantity means we don't know how many items were ordered, versus a quantity of zero means there were none ordered. If missing values are treated like zeroes, the totals could be misunderstood, since we'd assume that nothing was sold when, in reality, we just don't know. Conclusions about the actual sales could, therefore, be inaccurate if the calculations were incorrect in the first place.